# ENT Triage Model Finetuning

Finetune a small LLM (Llama 3) for ENT triage using synthetic transcript→output pairs.

**Output format:** SUMMARY, FINDINGS, FLAGS, URGENCY (routine, semi-urgent, or urgent), REASONING

**Deployment note:** Groq does *not* host custom finetuned models. After finetuning, deploy to **Together.ai**, **Replicate**, or **Ollama** (local).

In [ ]:
# Install dependencies (run once)
!pip install -q transformers peft datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.1 MB/s eta 0:00:00


In [ ]:
import json
from pathlib import Path
from datasets import Dataset
import torch

## 1. Load or upload training data

**Recommended (3+ sentence summaries):** Use `combined_triage_training.jsonl` (run `python modelling/code/prepare_training_data.py` first) or `triage_training_data_min3sentences.jsonl`.

- **Option A:** Upload your chosen file to Colab (e.g. `combined_triage_training.jsonl`).
- **Option B:** Use sample inline for a quick test.

In [ ]:
# Option A: Load from uploaded file
!pip install -q -U google-colab
from google.colab import files
uploaded = files.upload()  # Choose triage_training_data.jsonl
DATA_PATH = list(uploaded.keys())[0]

# Option B: Use sample inline
# SAMPLE_DATA = [
#     {
#         "transcript": "chief_complaint:mild sore throat. symptom_duration:3 days. symptom_severity:mild. symptom_progression:improving. red_flags:No. risk_factors:No",
#         "output": "SUMMARY: Patient presents with mild sore throat of 3 days duration. Symptoms improving. No red flags.\nFINDINGS:\n- Mild sore throat\n- 3-day duration\n- Improving trend\nFLAGS: [SYMPTOM] sore throat, [SEVERITY] mild, [DURATION] 3 days, [PROGRESSION] improving\nURGENCY: routine\nREASONING: Mild symptoms, improving, no red flags. Routine appointment."
#     },
#     {
#         "transcript": "chief_complaint:difficulty breathing. symptom_duration:few hours. symptom_severity:severe. symptom_progression:rapidly worsening. red_flags:Yes. risk_factors:No",
#         "output": "SUMMARY: Patient with acute difficulty breathing, rapidly worsening. Critical red flag.\nFINDINGS:\n- Difficulty breathing\n- Rapid deterioration\nFLAGS: [RED_FLAG] difficulty breathing, [SEVERITY] severe\nURGENCY: urgent\nREASONING: Difficulty breathing is a critical red flag. Same-day evaluation required."
#     },
# ]

def load_jsonl(path: str):
    data = []
    with open(path) as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Prefer combined (3+ sentence) data; fallback to min3sentences, training_data (instruction+input+output), or original
for candidate in ('combined_triage_training.jsonl', 'triage_training_data_min3sentences.jsonl', 'training_data.jsonl', 'triage_training_data.jsonl'):
    try:
        DATA_PATH = candidate
        data = load_jsonl(DATA_PATH)
        print(f"Loaded {len(data)} examples from {DATA_PATH}")
        break
    except FileNotFoundError:
        continue
else:
    data = SAMPLE_DATA
    print(f"Using {len(data)} sample examples (upload combined_triage_training.jsonl for full training)")

# Limit for quick run; set to None to use full dataset
MAX_EXAMPLES = None
if MAX_EXAMPLES is not None:
    data = data[:MAX_EXAMPLES]

# Oversample semi-urgent so model sees more examples per epoch
import re
def _parse_urgency(text):
    m = re.search(r'URGENCY:\s*(\w+(?:-\w+)?)', text, re.IGNORECASE)
    return m.group(1).lower() if m else None
semi_examples = [r for r in data if _parse_urgency(r.get("output", "")) == "semi-urgent"]
if semi_examples:
    data = data + semi_examples  # Repeat semi-urgent 2x total
    print(f"Oversampled {len(semi_examples)} semi-urgent examples (now {len(data)} total)")

Saving combined_triage_training.jsonl to combined_triage_training (1).jsonl
Loaded 2920 examples from combined_triage_training.jsonl
Oversampled 1360 semi-urgent examples (now 4280 total)


In [ ]:
# Convert to chat format for instruction tuning
# Handles: (instruction, input, output) e.g. training_data.jsonl | (transcript, output) e.g. combined_triage_training
def to_chat(r):
    if "instruction" in r and "input" in r:
        user_content = f"{r['instruction']}\n\n{r['input']}"
    else:
        user_content = f"""You are an ENT triage expert. Analyze this transcript and produce a structured triage output.
Include: (1) SUMMARY: 3+ sentence clinical summary, (2) FINDINGS: bullet points, (3) FLAGS: tagged items, (4) URGENCY: routine, semi-urgent, or urgent, (5) REASONING: brief justification. Prioritize red flags.

URGENCY must be exactly one of: routine, semi-urgent, or urgent. Use semi-urgent when: moderate severity + worsening, or fever + non-critical symptoms, or ear discharge, or symptoms needing evaluation within 24–48 hours.

TRANSCRIPT:
{r['transcript']}"""
    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": r["output"]},
        ]
    }

chat_data = [to_chat(r) for r in data]
dataset = Dataset.from_list(chat_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 3852
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 428
    })
})


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"  # Updated to Llama 3

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-69e53157-7f87fbb5736ee1b825eb5bbc;24dfe064-8871-4fa7-8efe-de252bde7f1d)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

Please add your Hugging Face token as a Colab secret named `HF_TOKEN` to access the gated model. You can create a token in your Hugging Face profile settings.

In [ ]:
# Used to securely store your API key
from google.colab import userdata

HF_TOKEN=userdata.get('HF_TOKEN')

Now, let's retry loading the model with the provided Hugging Face token.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"  # Updated to Llama 3

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-69e532db-2abce74729dc2bdf2780d9c5;9b901212-c640-42e4-8771-044e22a1359d)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Used to securely store your API key
from google.colab import userdata

HF_TOKEN=userdata.get('HF_TOKEN')

SecretNotFoundError: Secret HF_TOKEN does not exist.

Now, let's retry loading the model with the provided Hugging Face token.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # Changed to Mistral-7B-Instruct-v0.2

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [ ]:
from transformers import TrainingArguments, Trainer

# Ensure pad token is set
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

def tokenize(examples):
    out = tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
        return_tensors=None,
    )
    # Causal LM needs labels; use input_ids and mask padding with -100 so loss is not computed on padding
    out["labels"] = [
        [tid if tid != tokenizer.pad_token_id else -100 for tid in ids]
        for ids in out["input_ids"]
    ]
    return out

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
train_ds = train_ds.with_format("torch")

training_args = TrainingArguments(
    output_dir="./triage_mistral_lora", # Changed from triage_llama3_lora
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)
trainer.train()

Map:   0%|          | 0/3852 [00:00<?, ? examples/s]

Map:   0%|          | 0/3852 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.137962
20,1.466923
30,0.985547
40,0.543457
50,0.331411
60,0.258693
70,0.210913
80,0.168955
90,0.164261
100,0.144310


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=1446, training_loss=0.08830323516498133, metrics={'train_runtime': 5364.6194, 'train_samples_per_second': 2.154, 'train_steps_per_second': 0.27, 'total_flos': 5.058251584186614e+17, 'train_loss': 0.08830323516498133, 'epoch': 3.0})

In [ ]:
# Save LoRA adapter
trainer.save_model("./triage_mistral_lora") # Changed from triage_llama3_lora
tokenizer.save_pretrained("./triage_mistral_lora") # Changed from triage_llama3_lora

print("Saved to ./triage_mistral_lora. Download and deploy to Together.ai, Replicate, or Ollama.")

Saved to ./triage_mistral_lora. Download and deploy to Together.ai, Replicate, or Ollama.


## Evaluate finetuned model

In [ ]:
import re

def parse_urgency(text):
    m = re.search(r'URGENCY:\s*(\w+(?:-\w+)?)', text, re.IGNORECASE)
    return m.group(1).lower() if m else None

# Evaluate on test set (sample for speed; set EVAL_N to None for full test)
EVAL_N = 20
test_data = dataset["test"]
if EVAL_N is not None:
    test_data = test_data.select(range(min(EVAL_N, len(test_data))))

model.eval()
correct, total = 0, 0
results = []

for i, ex in enumerate(test_data):
    user_only = [{"role": "user", "content": ex["messages"][0]["content"]}]
    prompt = tokenizer.apply_chat_template(
        user_only, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    pred_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    expected = ex["messages"][1]["content"]
    pred_urg = parse_urgency(pred_text)
    exp_urg = parse_urgency(expected)
    if pred_urg and exp_urg:
        correct += pred_urg == exp_urg
        total += 1
    results.append((ex["messages"][0]["content"][:80] + "...", expected, pred_text))

if total > 0:
    print(f"URGENCY accuracy: {correct}/{total} = {100*correct/total:.1f}%")
print("\n--- Sample comparisons (User truncated | Expected | Predicted) ---")
for i, (user_preview, exp, pred) in enumerate(results[:5]):
    print(f"\n--- Example {i+1} ---")
    print(f"User: {user_preview}")
    print(f"Expected:\n{exp[:300]}...")
    print(f"Predicted:\n{pred[:300]}...")

In [ ]:
# Merge LoRA into base model
merged = model.merge_and_unload()
merged.save_pretrained("./triage_mistral_merged") # Changed from triage_llama3_merged
tokenizer.save_pretrained("./triage_mistral_merged") # Changed from triage_llama3_merged

# Download the merged folder to your Mac (e.g. via Colab Files or Drive)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./triage_mistral_merged/tokenizer_config.json',
 './triage_mistral_merged/chat_template.jinja',
 './triage_mistral_merged/tokenizer.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2") # Changed from meta-llama/Meta-Llama-3-8B-Instruct
model = PeftModel.from_pretrained(base, "./triage_mistral_lora") # Changed from triage_llama3_lora
merged = model.merge_and_unload()
merged.save_pretrained("./triage_mistral_merged") # Changed from triage_llama3_merged
tokenizer = AutoTokenizer.from_pretrained("./triage_mistral_lora") # Changed from triage_llama3_lora
tokenizer.save_pretrained("./triage_mistral_merged") # Changed from triage_llama3_merged

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./triage_mistral_merged/tokenizer_config.json',
 './triage_mistral_merged/chat_template.jinja',
 './triage_mistral_merged/tokenizer.json')

In [ ]:
!zip -r triage_mistral_merged.zip triage_mistral_merged

  adding: triage_mistral_merged/ (stored 0%)
  adding: triage_mistral_merged/model.safetensors


zip error: Interrupted (aborting)


In [ ]:
!ls -lh

In [ ]:
# download
from google.colab import files
files.download("triage_mistral_merged.zip")

## Push to Hugging Face Hub

If you want to push your finetuned model to the Hugging Face Hub, you can use the `push_to_hub` method. You will need to specify a repository ID. For example, `your_username/triage_mistral_finetuned`.

In [ ]:
import os
repo_id = "jmatni6/triage_mistral_finetuned"  # Change this to your desired repository ID

_hf_token = os.environ.get("HF_TOKEN")
if not _hf_token:
    raise RuntimeError("Set HF_TOKEN in the environment before push_to_hub (do not hardcode tokens in notebooks).")
merged.push_to_hub(repo_id, token=_hf_token)
tokenizer.push_to_hub(repo_id, token=_hf_token)

print(f"Model and tokenizer pushed to Hugging Face Hub: https://huggingface.co/{repo_id}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mcq41p4/model.safetensors:   0%|          | 24.8kB / 14.5GB            

Model and tokenizer pushed to Hugging Face Hub: https://huggingface.co/jmatni6/triage_mistral_finetuned


## Inference with the finetuned model

Now you can load your finetuned model and tokenizer from the Hugging Face Hub and use it to make predictions. Make sure `repo_id` is correctly set to your model's repository.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Replace with your actual repository ID if different
repo_id = "jmatni6/triage_mistral_finetuned"

# Load the finetuned model and tokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
inference_model = AutoModelForCausalLM.from_pretrained(repo_id, token=HF_TOKEN)

inference_model.eval() # Set model to evaluation mode


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

Now, let's test it with an example transcript.

In [ ]:
import torch

# Example transcript for inference
example_transcript = "chief_complaint:Patient reports sudden onset of severe headache. symptom_duration:2 hours. symptom_severity:severe. symptom_progression:worsening. red_flags:Yes, stiff neck, photophobia. risk_factors:No"

# Create the chat prompt format
user_prompt = f"""You are an ENT triage expert. Analyze this transcript and produce a structured triage output.\nInclude: (1) SUMMARY: 3+ sentence clinical summary, (2) FINDINGS: bullet points, (3) FLAGS: tagged items, (4) URGENCY: routine, semi-urgent, or urgent, (5) REASONING: brief justification. Prioritize red flags.\n\nURGENCY must be exactly one of: routine, semi-urgent, or urgent. Use semi-urgent when: moderate severity + worsening, or fever + non-critical symptoms, or ear discharge, or symptoms needing evaluation within 24–48 hours.\n\nTRANSCRIPT:\n{example_transcript}"""

messages = [
    {"role": "user", "content": user_prompt}
]

# Apply chat template and tokenize
input_prompt = inference_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = inference_tokenizer(input_prompt, return_tensors="pt").to(inference_model.device)

# Generate output
with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=512, # Adjust max_new_tokens as needed
        do_sample=False, # For deterministic output
        pad_token_id=inference_tokenizer.eos_token_id
    )

# Decode and print the generated text
generated_text = inference_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated Triage Output:")
print(generated_text)

Generated Triage Output:
SUMMARY: Patient reports sudden onset of severe headache with rapid onset; symptoms are worsening. This represents a potential red flag for serious etiology.
FINDINGS:
- sudden onset of severe headache
- Rapid evaluation needed
FLAGS: [RED_FLAG] severe headache, [SEVERITY] severe, [PROGRESSION] worsening
URGENCY: semi-urgent
REASONING: Moderate severity with worsening symptoms. Should be seen within 24-48 hours.


## Deploy Options (Groq does NOT host custom models)

1. **Together.ai** – Upload merged model or LoRA; use their inference API.
2. **Replicate** – Package model as a Replicate model; call via their API.
3. **Ollama** – Convert to GGUF, run locally: `ollama create triage -f Modelfile`.